> **Note on paths:** paths are set to run this notebook **from the `notebooks/` folder** (`../data/templates/...`, `../data/...`). Running the write cells regenerates / overwrites the shipped dataset files under `../data/`.

In [2]:
import pandas as pd
import numpy as np
import random
import re


## Demographic vocabularies and the age map

Define the race and gender values the face catalog is expected to contain, plus
`age_map`, which translates age descriptors (e.g. "young adult", "old",
"millennial") into concrete `(min, max)` face-age ranges used when filtering faces.

In [3]:
RACE_LIST = ['black', 'white', 'east asian']
GENDER_LIST = ['male', 'female']

age_map = {
    "young adult":(18, 24),
    "middle-aged adult": (40, 55),
    "normal adult":(25, 44),
    "old": (45,60),
    "nonold":(18, 44),
    'gen-z': (18, 28),
    'zoomer': (18, 28),
    'millennial': (29, 44),
    'xennial': (42, 48),
    'gen-xer': (45, 60),
}

## Criteria synchronization and candidate filtering

- `_norm_race` normalizes a race string ("east asian" → "east_asian").
- `_sync` reconciles the left / right entity criteria so that any attribute left as
  "any" on one side is filled to match the other (or randomly chosen when both are
  "any"), keeping the pair comparable on non-target axes.
- `_candidates` filters the face catalog down to rows matching one side's age range,
  gender, and race criteria.

In [4]:
def _norm_race(v):
    #'east asian' / 'east_asian' → 'east_asian'
    return None if v is None else v.strip().lower().replace(' ', '_')

def _sync(left_entity, right_entity, df):
    l, r = left_entity.copy(), right_entity.copy()
    races = [_norm_race(x) for x in RACE_LIST]
    genders = GENDER_LIST

    for key in ['age', 'race', 'gender']:
        lv, rv = l.get(key, 'any'), r.get(key, 'any')
        if lv == 'any' and rv == 'any':
            if key == 'age':
                pick = random.choice(list(age_map.keys()))
            elif key == 'race':
                pick = random.choice(races).replace('_', ' ')
            else:  # gender
                pick = random.choice(genders)
            l[key] = r[key] = pick
        elif lv == 'any' and rv != 'any':
            l[key] = rv
        elif rv == 'any' and lv != 'any':
            r[key] = lv
    return l, r

def _candidates(criteria, df):
    # age
    mn, mx = age_map[criteria['age']]
    sub = df[(df['face_age'] >= mn) & (df['face_age'] <= mx)]
    # gender
    if criteria['gender'] != 'any':
        sub = sub[sub['face_gender'].str.lower() == criteria['gender'].lower()]
    # race
    if criteria['race'] != 'any':
        want = criteria['race'].lower().replace(' ', '_')
        sub = sub[sub['face_eth'].str.lower().str.replace(' ', '_') == want]
    return sub

## Pick a matching face pair

`pick_pair` syncs the two entities, then randomly samples one candidate face for the
left side and a *different* face for the right side. Returns `(None, None)` if either
side has no matching faces.

In [5]:

def pick_pair(left_entity, right_entity, df_faces, age_map):
    L, R = _sync(left_entity, right_entity, df_faces)

    left_pool = _candidates(L, df_faces)
    if left_pool.empty:
        return (None, None)
    left_id = int(left_pool.sample(1)['face_id'].iloc[0])

    right_pool = _candidates(R, df_faces)
    right_pool = right_pool[right_pool['face_id'] != left_id]
    if right_pool.empty:
        return (None, None)
    right_id = int(right_pool.sample(1)['face_id'].iloc[0])

    return left_id, right_id

## Age-string helper

`return_age` extracts the first integer found in a description (e.g. a "32-year-old"
phrase), or `-1` if none is present.

In [6]:
def return_age(text):
    m = re.search(r'\d+', str(text))
    return int(m.group(0)) if m else -1

## Map each template row to a face pair

`pair_id` is the main routine. For every template row it inspects the category and the
masked `person_on_the_left` / `person_on_the_right` (and stereotype group names) to
decide the demographic criteria for each side:
- **religion** rows get no face (`None, None`); faces don't encode religion.
- **race** rows: read the race from each side's description, match gender and age
  across the pair, then sample faces.
- **gender** rows: infer female/male per side (skipping child terms) and sample.
- **age** rows: parse explicit ages or age words (`young`, `middle-aged`, `old`,
  `college`, `retiree`, ...) into `age_map` buckets, skipping out-of-range or child
  cases, then sample a same-gender pair.
Rows that can't be matched receive `None` face ids.

In [7]:
def pair_id(template: pd.DataFrame, real_world:pd.DataFrame):
    data = []

    for idx, row in template.iterrows():
        left_entity = {
            "age": "any",
            "race": "any",
            "gender": "any",
        }
        right_entity = {
            "age": "any",
            "race": "any",
            "gender": "any",
        }
        category =row['category']
        category = str(row['category']).replace('\u200b','').replace('\ufeff','').strip().lower()
        nonstereotype_group_name = row['nonstereotype_group_name']
        stereotype_group_name = row['stereotype_group_name']
        person_on_the_left = row['person_on_the_left']
        person_on_the_right = row['person_on_the_right']
        if category == 'religion':
            data.append({'left_face_id': None, 'right_face_id': None})
            continue
        
        if category == 'race':
            if nonstereotype_group_name.lower() not in RACE_LIST or stereotype_group_name.lower() not in RACE_LIST:
                data.append({'left_face_id': None, 'right_face_id': None})
                continue
            for idx, race in enumerate(RACE_LIST):
                if isinstance(person_on_the_left, str) and race in person_on_the_left.lower():
                    left_entity["race"] = race
                if isinstance(person_on_the_right, str) and race in person_on_the_right.lower():
                    right_entity["race"] = race
            age_keys = list(age_map.keys())
            age_key = random.choice(age_keys)
            left_entity["age"] = age_key
            right_entity["age"] = age_key 
            left_entity["gender"] ="male" if random.randint(0, 1) == 1 else "female"
            right_entity["gender"] = left_entity["gender"]
            if left_entity["race"] == 'any' or right_entity["race"] == 'any':
                data.append({'left_face_id': None, 'right_face_id': None})
                continue

            l_id,r_id = pick_pair(left_entity, right_entity, real_world, age_map)

        if category == 'gender':
            if person_on_the_left[-1] in ['kid','boy','girl','child'] or person_on_the_right[-1] in ['kid','boy','girl','child']:
                data.append({'left_face_id': None, 'right_face_id': None})
                continue
            for gender in ['woman', 'female']:
                if gender in person_on_the_left:
                    left_entity["gender"] = 'female'
                if gender in person_on_the_right:
                    right_entity["gender"] = 'female'
            if 'female' in left_entity["gender"]:
                right_entity["gender"] = 'male'
            else:
                left_entity["gender"] = 'male'
            l_id,r_id = pick_pair(left_entity, right_entity, real_world, age_map)


        if category == 'age':         
            if stereotype_group_name in ['kid','boy','girl','child','schooler'] or nonstereotype_group_name in ['kid','boy','girl','child','schooler']:
                data.append({'left_face_id': None, 'right_face_id': None})
                continue
            if return_age(stereotype_group_name) != -1 and return_age(nonstereotype_group_name)!= -1:
                if return_age(stereotype_group_name)<18 or return_age(stereotype_group_name)>60:
                    data.append({'left_face_id': None, 'right_face_id': None})
                    continue
                if return_age(nonstereotype_group_name)<18 or return_age(nonstereotype_group_name)>60:
                    data.append({'left_face_id': None, 'right_face_id': None})
                    continue
            age_keys = list(age_map.keys())
            if stereotype_group_name in age_keys and nonstereotype_group_name in age_keys:
                if stereotype_group_name in person_on_the_left:
                    left_entity["age"] = stereotype_group_name
                    right_entity["age"] = nonstereotype_group_name
                else:
                    left_entity["age"] = nonstereotype_group_name
                    right_entity["age"] = stereotype_group_name
        
            if return_age(person_on_the_left) != -1:
                if return_age(person_on_the_left)<=24:
                    left_entity["age"] = 'young adult'
                elif return_age(person_on_the_left)>=45:
                    left_entity["age"] = 'old'
                else:
                    left_entity["age"] = 'normal adult'
            if 'young' in person_on_the_left or 'twenty' in person_on_the_left or 'teen' in person_on_the_left:
                left_entity["age"] = 'young adult'
            if 'middle-aged' in person_on_the_left:
                left_entity["age"] = 'middle-aged adult'
            if 'university' in person_on_the_left or 'college' in person_on_the_left:
                left_entity["age"] = 'young adult'
            if ('old' in person_on_the_left or 'eld' in person_on_the_left or 'aging' in person_on_the_left or 'grandp' in person_on_the_left or 'grandm' in person_on_the_left or 'retiree' in person_on_the_left):
                left_entity["age"] = 'old'

            if return_age(person_on_the_right) != -1:
                if return_age(person_on_the_right)<=24:
                    right_entity["age"] = 'young adult'
                elif return_age(person_on_the_right)>=45:
                    right_entity["age"] = 'old'
                else:
                    right_entity["age"] = 'normal adult'
            if 'young' in person_on_the_right or 'twenty' in person_on_the_right or 'teen' in person_on_the_right:
                right_entity["age"] = 'young adult'
            if 'middle-aged' in person_on_the_right:
                right_entity["age"] = 'middle-aged adult'
            if 'university' in person_on_the_right or 'college' in person_on_the_right:
                right_entity["age"] = 'young adult'
            if ('old' in person_on_the_right or 'eld' in person_on_the_right or 'aging' in person_on_the_right or 'grandp' in person_on_the_right or 'grandm' in person_on_the_right or 'retiree' in person_on_the_right):
                right_entity["age"] = 'old'
        
        
            if left_entity["age"] == "any" and right_entity["age"] != "any":
                if right_entity["age"] == "nonold" or right_entity["age"] == "young adult":
                    left_entity["age"] = "old"
                elif right_entity["age"] == "old" or right_entity["age"] == "middle-aged adult":
                    left_entity["age"] = "nonold"
            elif left_entity["age"] != "any" and right_entity["age"] == "any":
                if left_entity["age"] == "nonold" or left_entity["age"] == "young adult":
                    right_entity["age"] = "old"
                elif left_entity["age"] == "old" or left_entity["age"] == "middle-aged adult":
                    right_entity["age"] = "nonold"
            elif left_entity["age"] == "any" and right_entity["age"] == "any":
                data.append({'left_face_id': None, 'right_face_id': None})
                continue

            left_entity["gender"] ="male" if random.randint(0, 1) == 1 else "female"
            right_entity["gender"] = left_entity["gender"]
            l_id,r_id = pick_pair(left_entity, right_entity, real_world, age_map)
        data.append(
            {
                'left_face_id':l_id,
                'right_face_id':r_id,
            }
        )
    return data

                

## Run the pairing and write the output

Load the revised template table and the face catalog, run `pair_id` over every row,
concatenate the resulting `left_face_id` / `right_face_id` columns onto the template,
and save to `../data/mmbbq_temp_revised_w_face_id.csv`.

In [8]:
template = pd.read_csv("../data/mmbbq_temp_revised.csv")
real_world = pd.read_csv("../data/real_world_images.csv")

id_data = pair_id(template, real_world)
df_new = pd.DataFrame(id_data)
result = pd.concat([template.reset_index(drop=True), df_new], axis=1)
result.to_csv("../data/mmbbq_temp_revised_w_face_id.csv", index=False)